# OrthoDB — Hierarchical Catalogue of Eukaryotic Orthologs

**OrthoDB** is a comprehensive catalogue of orthologous protein-coding genes across vertebrates, arthropods, fungi, plants, and other eukaryotes, as well as bacteria. It provides a hierarchical system of orthologs defined at each node of the species phylogeny, grouping genes that descended from a single ancestral gene in the last common ancestor of the compared species.

Orthologs are genes in different species that evolved from a common ancestral gene by speciation (as opposed to paralogs, which arise by gene duplication). Identifying orthologs is fundamental for functional annotation transfer, comparative genomics, and evolutionary studies.

| Property | Value |
|---|---|
| **Current version** | v12 (odb12v1) |
| **Number of species** | ~1,300+ eukaryotes and bacteria |
| **Number of orthogroups** | ~1.5 million hierarchical orthogroups |
| **URL** | https://www.orthodb.org |
| **Bulk downloads** | https://data.orthodb.org/download/ |
| **Primary use cases** | Comparative genomics, gene function transfer, single-copy universal orthologs (e.g. BUSCO markers), phylogenetics |

**Reference:** Kuznetsov et al. (2023), *Nucleic Acids Research*, OrthoDB v11

In [ ]:
import requests
import time
import gzip
import json
from pathlib import Path

import polars as pl

# TODO

* [x] **Ingest data**
    * [x] Connect to OrthoDB REST API and confirm access
    * [x] Search API for a model gene (e.g. TP53) to retrieve orthogroup IDs
    * [x] Download bulk orthogroup tables (`odb12v1_OGs.tab.gz`, `odb12v1_OG2genes.tab.gz`) from OrthoDB with caching
    * [x] Parse tab-delimited files into Polars DataFrames with correct dtypes
    * [x] Save to `data/` with caching
* [ ] **Explore and clean**
    * [ ] Summarise DataFrame dimensions (orthogroups, genes, species)
    * [ ] Check for missing values and duplicate orthogroup IDs
    * [ ] Map NCBI taxon IDs to species names
    * [ ] Filter orthogroups to vertebrates only
* [ ] **Analysis**
    * [ ] Compute orthogroup size distribution (genes per group)
    * [ ] Identify single-copy universal orthologs
    * [ ] Compare species representation per clade
* [ ] **Visualization**
    * [ ] Plot orthogroup size distribution (histogram)
    * [ ] Heatmap of species × clade representation
* [ ] **Statistical analysis**
    * [ ] Test for deviations from expected gene counts per clade

## 1. Ingest Data

### 1.1 Connect to OrthoDB API

In [ ]:
ORTHODB_BASE = "https://www.orthodb.org"
SPECIES_HUMAN = 9606  # NCBI taxon ID for Homo sapiens

def orthodb_get(endpoint: str, params: dict) -> requests.Response:
    """
    Send a GET request to the OrthoDB REST API.

    Parameters
    ----------
    endpoint : str
        API path (e.g. "search").
    params : dict
        Query parameters.

    Returns
    -------
    requests.Response
        Raw response object; caller is responsible for parsing.
    """
    url = f"{ORTHODB_BASE}/{endpoint}"
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    time.sleep(0.5)  # be polite to the server
    return resp

# Search for TP53 orthogroups in Homo sapiens
resp = orthodb_get("search", {
    "query": "TP53",
    "ncbi_tax_id": SPECIES_HUMAN,
    "limit": 10,
})

data = resp.json()
print(f"Status: {resp.status_code}")
print(f"Response keys: {list(data.keys())}")
print(f"\nFirst result:")
print(json.dumps(data["data"][0] if data.get("data") else data, indent=2))

### 1.2 Download Bulk Orthogroup Tables

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

ORTHODB_DOWNLOAD_BASE = "https://data.orthodb.org/download"

# Files to download
BULK_FILES = {
    "odb12v1_OGs.tab.gz": "Orthogroup table (og_id, level_taxid, og_name)",
    "odb12v1_OG2genes.tab.gz": "Orthogroup-to-gene mapping (og_id, gene_id)",
    "odb12v1_genes.tab.gz": "Gene descriptions and metadata",
}

def download_with_cache(filename: str, description: str) -> Path:
    """
    Download a file from OrthoDB bulk downloads to DATA_DIR, skipping if
    already present (simple file-existence cache).

    Parameters
    ----------
    filename : str
        Filename to download (e.g. "odb12v1_OGs.tab.gz").
    description : str
        Human-readable description for progress messages.

    Returns
    -------
    Path
        Local path to the downloaded (or cached) file.
    """
    local_path = DATA_DIR / filename
    if local_path.exists():
        print(f"[cache] {filename}  ({description})")
        return local_path

    url = f"{ORTHODB_DOWNLOAD_BASE}/{filename}"
    print(f"Downloading {filename}  ({description}) ...")
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB", end="\r")
    print(f"\nSaved to {local_path}")
    return local_path

# Download all three bulk files
paths = {name: download_with_cache(name, desc) for name, desc in BULK_FILES.items()}
print("\nAll files ready.")

### 1.3 Parse into Polars DataFrames

In [ ]:
# ── Orthogroups table: odb12v1_OGs.tab.gz ─────────────────────────────────────
# Columns: og_id (str), level_taxid (i64), og_name (str)
# Each row is one orthogroup anchored at a specific taxonomic level.
# og_id format: "<int>at<taxid>"  e.g. "1234at9606"
ogs = pl.read_csv(
    paths["odb12v1_OGs.tab.gz"],
    separator="\t",
    has_header=False,
    new_columns=["og_id", "level_taxid", "og_name"],
    schema_overrides={"level_taxid": pl.Int64},
)

# ── OG-to-gene mapping: odb12v1_OG2genes.tab.gz ───────────────────────────────
# Columns: og_id (str), gene_id (str)
# Each row maps one gene (in OrthoDB's internal gene ID format) to one orthogroup.
# A gene may appear in multiple orthogroups at different taxonomic levels.
og2genes = pl.read_csv(
    paths["odb12v1_OG2genes.tab.gz"],
    separator="\t",
    has_header=False,
    new_columns=["og_id", "gene_id"],
)

# ── Genes table: odb12v1_genes.tab.gz ─────────────────────────────────────────
# Columns: gene_id, organism_taxid, protein_id, uniprot_id, gene_name, ncbi_gene_id, description
genes = pl.read_csv(
    paths["odb12v1_genes.tab.gz"],
    separator="\t",
    has_header=False,
    new_columns=[
        "gene_id", "organism_taxid", "protein_id",
        "uniprot_id", "gene_name", "ncbi_gene_id", "description",
    ],
    schema_overrides={"organism_taxid": pl.Int64, "ncbi_gene_id": pl.Int64},
    infer_schema_length=10_000,
    ignore_errors=True,
)

print("Parsed all three tables successfully.")

### 1.4 Quick DataFrame Summary

In [ ]:
for name, df in [("ogs", ogs), ("og2genes", og2genes), ("genes", genes)]:
    print(f"{'='*60}")
    print(f"DataFrame: {name}")
    print(f"  Shape  : {df.shape[0]:>10,} rows × {df.shape[1]} columns")
    print(f"  Dtypes : {dict(zip(df.columns, [str(t) for t in df.dtypes]))}")
    print()
    print(df.head(5))
    print()